# 🚀 Titan-GPT V3 Training on Google Colab

This notebook trains the Titan-GPT V3 model on GPU and saves checkpoints to Google Drive.

## Features:
- ✅ GPU acceleration (T4/V100/A100)
- ✅ Automatic Google Drive mounting
- ✅ Checkpoint saving to Drive
- ✅ Progress monitoring
- ✅ Result visualization

## Setup Requirements:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Run all cells in order
3. Authorize Google Drive access when prompted

## 📋 Step 1: Check GPU Availability

In [ ]:
import os
import torch

# Check GPU
if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ GPU not available! Please enable GPU:")
    print("   Runtime → Change runtime type → Hardware accelerator → GPU")

## 💾 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create directory for checkpoints
CHECKPOINT_DIR = '/content/drive/MyDrive/Titan_V3_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"📁 Checkpoint directory: {CHECKPOINT_DIR}")

## 📦 Step 3: Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
if not os.path.exists('/content/LLMs-from-scratch'):
    print("📥 Cloning repository...")
    !git clone --depth 1 https://github.com/rasbt/LLMs-from-scratch.git
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

# Change to repo directory
%cd /content/LLMs-from-scratch

In [ ]:
# Install dependencies
print("📦 Installing dependencies...")
!pip install -q torch tiktoken matplotlib tqdm numpy
print("✅ Dependencies installed")

## 🏗️ Step 4: Setup Model Files (if not in repo)

In [ ]:
# Create titan-optimal directory structure if needed
os.makedirs('/content/LLMs-from-scratch/titan-optimal/models', exist_ok=True)
os.makedirs('/content/LLMs-from-scratch/titan-optimal/configs', exist_ok=True)
os.makedirs('/content/LLMs-from-scratch/titan-optimal/checkpoints', exist_ok=True)

print("✅ Directory structure created")

# Note: If the titan-optimal files don't exist in the repo, 
# you'll need to upload them or copy from your local setup

## 🎯 Step 5: Create GPU-Optimized Training Script

In [ ]:
%%writefile /content/LLMs-from-scratch/titan-optimal/train_v3_colab.py
"""GPU-Optimized Training for Titan-GPT V3 on Google Colab

Features:
- GPU acceleration
- Google Drive checkpoint saving
- Progress visualization
- Memory-efficient training
"""

import os
import sys
import torch
import torch.nn as nn
import time
import json
from pathlib import Path
from tqdm import tqdm

# Add paths
sys.path.append('/content/LLMs-from-scratch')
sys.path.append('/content/LLMs-from-scratch/ch04/01_main-chapter-code')

from titan-optimal.models.titan_gpt_v3 import TitanGPTModelV3
from titan-optimal.configs.model_configs_v3 import get_model_config_v3, get_training_config_v3
from gpt import create_dataloader_v1

def train_titan_v3_colab(
    model_size="small",
    checkpoint_dir="/content/drive/MyDrive/Titan_V3_Checkpoints",
    num_epochs=10,
    batch_size=8,
    save_every_n_epochs=2
):
    """Train Titan-GPT V3 on Colab GPU."""
    
    print("="*80)
    print("TITAN-GPT V3 TRAINING ON GOOGLE COLAB")
    print("="*80)
    
    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    torch.manual_seed(123)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(123)
    
    # Load training data
    print("\n📥 Loading training data...")
    data_path = "/content/LLMs-from-scratch/ch05/01_main-chapter-code/the-verdict.txt"
    
    if not os.path.exists(data_path):
        import requests
        print("Downloading data...")
        url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
        response = requests.get(url, timeout=30)
        os.makedirs(os.path.dirname(data_path), exist_ok=True)
        with open(data_path, "w", encoding="utf-8") as f:
            f.write(response.text)
    
    with open(data_path, "r", encoding="utf-8") as f:
        text_data = f.read()
    
    print(f"✅ Loaded {len(text_data)} characters")
    
    # Split data
    train_ratio = 0.90
    split_idx = int(train_ratio * len(text_data))
    
    # Get configuration
    train_cfg = get_training_config_v3(model_size)
    train_cfg["batch_size"] = batch_size
    train_cfg["num_epochs"] = num_epochs
    
    # GPU optimizations
    if device.type == "cuda":
        train_cfg["batch_size"] = min(batch_size * 2, 16)  # Larger batches for GPU
        train_cfg["gradient_accumulation_steps"] = max(train_cfg["gradient_accumulation_steps"] // 2, 1)
    
    context_length = 512 if model_size == "small" else 1024
    
    # Create dataloaders
    print("\n📊 Creating dataloaders...")
    train_loader = create_dataloader_v1(
        text_data[:split_idx],
        batch_size=train_cfg["batch_size"],
        max_length=context_length,
        stride=context_length,
        drop_last=True,
        shuffle=True,
        num_workers=0
    )
    
    val_loader = create_dataloader_v1(
        text_data[split_idx:],
        batch_size=train_cfg["batch_size"],
        max_length=context_length,
        stride=context_length,
        drop_last=False,
        shuffle=False,
        num_workers=0
    )
    
    print(f"Train batches: {len(train_loader)}")
    print(f"Val batches: {len(val_loader)}")
    
    # Create model
    print(f"\n🏗️ Creating Titan-GPT V3 {model_size.upper()} model...")
    v3_cfg = get_model_config_v3(model_size, use_memory=True)
    v3_cfg["batch_size"] = train_cfg["batch_size"]
    
    model = TitanGPTModelV3(v3_cfg).to(device)
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=train_cfg["learning_rate"],
        weight_decay=train_cfg["weight_decay"]
    )
    
    # Training setup
    print("\n" + "="*80)
    print("STARTING TRAINING")
    print("="*80)
    print(f"Epochs: {num_epochs}")
    print(f"Batch size: {train_cfg['batch_size']}")
    print(f"Gradient accumulation: {train_cfg['gradient_accumulation_steps']}")
    print(f"Learning rate: {train_cfg['learning_rate']}")
    print(f"Checkpoints: {checkpoint_dir}")
    print("="*80)
    
    # Training loop
    train_losses = []
    val_losses = []
    perplexities = []
    best_val_loss = float('inf')
    
    model.train()
    
    for epoch in range(num_epochs):
        print(f"\n{'='*80}")
        print(f"EPOCH {epoch+1}/{num_epochs}")
        print(f"{'='*80}")
        
        epoch_start = time.time()
        epoch_loss = 0.0
        
        # Training loop with progress bar
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch_idx, (input_batch, target_batch) in enumerate(pbar):
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            # Forward pass
            logits, aux_losses = model(
                input_batch,
                targets=target_batch,
                update_memory=True,
                mode="train"
            )
            
            # Loss
            loss = nn.functional.cross_entropy(
                logits.flatten(0, 1),
                target_batch.flatten()
            )
            
            # Backward
            loss = loss / train_cfg["gradient_accumulation_steps"]
            loss.backward()
            
            epoch_loss += loss.item()
            
            # Update weights
            if (batch_idx + 1) % train_cfg["gradient_accumulation_steps"] == 0:
                optimizer.step()
                optimizer.zero_grad()
            
            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for input_batch, target_batch in val_loader:
                input_batch = input_batch.to(device)
                target_batch = target_batch.to(device)
                
                logits, _ = model(
                    input_batch,
                    update_memory=False,
                    mode="inference"
                )
                
                val_loss += nn.functional.cross_entropy(
                    logits.flatten(0, 1),
                    target_batch.flatten()
                ).item()
        
        val_loss /= len(val_loader)
        train_loss = epoch_loss / len(train_loader)
        perplexity = torch.exp(torch.tensor(val_loss)).item()
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        perplexities.append(perplexity)
        
        epoch_time = time.time() - epoch_start
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}")
        print(f"  Perplexity: {perplexity:.2f}")
        print(f"  Time: {epoch_time:.2f}s")
        
        # Save checkpoint
        if (epoch + 1) % save_every_n_epochs == 0 or val_loss < best_val_loss:
            checkpoint_path = os.path.join(
                checkpoint_dir,
                f"titan_v3_{model_size}_epoch_{epoch+1}.pth"
            )
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'perplexity': perplexity,
            }, checkpoint_path)
            print(f"  💾 Checkpoint saved: {checkpoint_path}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_checkpoint_path = os.path.join(
                    checkpoint_dir,
                    f"titan_v3_{model_size}_best.pth"
                )
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'train_loss': train_loss,
                    'val_loss': val_loss,
                    'perplexity': perplexity,
                }, best_checkpoint_path)
                print(f"  ⭐ Best model saved: {best_checkpoint_path}")
        
        model.train()
        
        # Reset short-term memory
        if hasattr(model, 'reset_memory'):
            model.reset_memory(level="short")
    
    # Save final results
    results = {
        "model_size": model_size,
        "total_params": total_params,
        "num_epochs": num_epochs,
        "batch_size": train_cfg["batch_size"],
        "final_train_loss": train_losses[-1],
        "final_val_loss": val_losses[-1],
        "final_perplexity": perplexities[-1],
        "best_val_loss": best_val_loss,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "perplexities": perplexities,
    }
    
    results_path = os.path.join(checkpoint_dir, f"training_results_{model_size}.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    
    print("\n" + "="*80)
    print("TRAINING COMPLETE")
    print("="*80)
    print(f"Model: Titan-GPT V3 {model_size.upper()}")
    print(f"Parameters: {total_params:,}")
    print(f"Best Val Loss: {best_val_loss:.4f}")
    print(f"Final Perplexity: {perplexities[-1]:.2f}")
    print(f"\n📁 All checkpoints saved to: {checkpoint_dir}")
    print("="*80)
    
    return results

if __name__ == "__main__":
    # Train the model
    results = train_titan_v3_colab(
        model_size="small",
        num_epochs=10,
        batch_size=8,
        save_every_n_epochs=2
    )


## 🎯 Step 6: Run Training

In [ ]:
# Import the training function
import sys
sys.path.append('/content/LLMs-from-scratch/titan-optimal')

from train_v3_colab import train_titan_v3_colab

# Configure training parameters
MODEL_SIZE = "small"  # Options: "small" (124M params) or "medium" (340M params)
NUM_EPOCHS = 10
BATCH_SIZE = 8  # Adjust based on GPU memory
SAVE_EVERY_N_EPOCHS = 2

# Run training
results = train_titan_v3_colab(
    model_size=MODEL_SIZE,
    checkpoint_dir=CHECKPOINT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    save_every_n_epochs=SAVE_EVERY_N_EPOCHS
)

## 📊 Step 7: Visualize Training Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training progress
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training Loss
axes[0].plot(results['train_losses'], marker='o', label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Loss
axes[1].plot(results['val_losses'], marker='o', color='orange', label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Validation Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Perplexity
axes[2].plot(results['perplexities'], marker='o', color='green', label='Perplexity')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Perplexity')
axes[2].set_title('Perplexity (Lower is Better)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

# Save plot to Google Drive
plot_path = f"{CHECKPOINT_DIR}/training_progress.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Plot saved to: {plot_path}")

plt.show()

# Print summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Model: Titan-GPT V3 {MODEL_SIZE.upper()}")
print(f"Total Parameters: {results['total_params']:,}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Final Train Loss: {results['final_train_loss']:.4f}")
print(f"Final Val Loss: {results['final_val_loss']:.4f}")
print(f"Best Val Loss: {results['best_val_loss']:.4f}")
print(f"Final Perplexity: {results['final_perplexity']:.2f}")
print("="*60)

## 💾 Step 8: Load and Test Best Checkpoint

In [ ]:
import torch

# Load best checkpoint
best_checkpoint_path = f"{CHECKPOINT_DIR}/titan_v3_{MODEL_SIZE}_best.pth"

if os.path.exists(best_checkpoint_path):
    print(f"📥 Loading best checkpoint: {best_checkpoint_path}")
    
    checkpoint = torch.load(best_checkpoint_path)
    
    print("\nCheckpoint Info:")
    print(f"  Epoch: {checkpoint['epoch']}")
    print(f"  Train Loss: {checkpoint['train_loss']:.4f}")
    print(f"  Val Loss: {checkpoint['val_loss']:.4f}")
    print(f"  Perplexity: {checkpoint['perplexity']:.2f}")
    
    # You can load the model state for inference
    # model.load_state_dict(checkpoint['model_state_dict'])
    # model.eval()
    
    print("\n✅ Best model ready for inference!")
else:
    print(f"⚠️ Checkpoint not found: {best_checkpoint_path}")

## 📁 Step 9: List All Saved Checkpoints

In [ ]:
# List all files in checkpoint directory
print("📁 Saved files in Google Drive:")
print("="*60)

for file in sorted(os.listdir(CHECKPOINT_DIR)):
    file_path = os.path.join(CHECKPOINT_DIR, file)
    file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
    print(f"  {file} ({file_size:.2f} MB)")

print("="*60)
print(f"\n✅ All files are saved in: {CHECKPOINT_DIR}")
print("You can access them from Google Drive → MyDrive → Titan_V3_Checkpoints")

## 🎉 Training Complete!

### What's Been Saved:
1. **Checkpoints**: Saved every N epochs in Google Drive
2. **Best Model**: Best performing model based on validation loss
3. **Training Results**: JSON file with all metrics
4. **Visualization**: Training progress plots

### Next Steps:
1. Download checkpoints from Google Drive
2. Use best model for inference/fine-tuning
3. Experiment with different hyperparameters
4. Try the medium model (340M params) for better performance

### Tips:
- For longer training sessions, consider Colab Pro for extended runtime
- Adjust batch size based on GPU memory (increase for better GPUs)
- Monitor GPU usage: `!nvidia-smi`
- Medium model requires more GPU memory (recommend V100 or A100)

## 🔧 Optional: Monitor GPU Usage

In [ ]:
# Check GPU memory usage
!nvidia-smi

## 📝 Configuration Reference

### Model Sizes:
- **Small (124M params)**:
  - GPU Memory: ~4-6 GB
  - Batch size: 8-16
  - Training time: ~30-45 min (T4 GPU)

- **Medium (340M params)**:
  - GPU Memory: ~10-15 GB
  - Batch size: 4-8
  - Training time: ~1-2 hours (V100 GPU)

### Recommended Settings by GPU:
- **T4 (16GB)**: Small model, batch_size=8-12
- **V100 (16GB)**: Small (batch_size=16) or Medium (batch_size=4)
- **A100 (40GB)**: Medium model, batch_size=8-12

### Troubleshooting:
- **Out of Memory**: Reduce batch_size or use smaller model
- **Slow Training**: Ensure GPU is enabled and being used
- **Drive Full**: Delete old checkpoints or increase Drive storage